# 🔁 Forward et Backward Propagation — Notebook Bootcamp Data Science

## Module Data Science — Deep Learning

> **Prérequis :** cours `09-DataScience_Forward_Backward_Propagation.md`.

Ce notebook rend le cours **exécutable** : on reproduit chaque calcul, puis on construit et on entraîne
un réseau de neurones **from scratch** en NumPy. Il est autonome (datasets scikit-learn, aucun CSV).

| Partie | Contenu |
|--------|---------|
| **1** | Le neurone et les fonctions d'activation |
| **2** | Forward propagation sur l'exemple « email » |
| **3** | La loss |
| **4** | Backpropagation pas à pas |
| **5** | Mise à jour et itération |
| **6** | Un réseau complet from scratch + gradient checking |
| **7** | Expériences : learning rate, problème de symétrie |
| **8** | Comparaison avec scikit-learn |

In [ ]:
!pip install scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Fonctions d'activation et leurs dérivées (utilisées en backpropagation)
def sigmoid(z):   return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def relu(z):      return np.maximum(0, z)
def d_relu(z):    return (z > 0).astype(float)
def d_tanh(z):    return 1 - np.tanh(z) ** 2

print('Environnement prêt ✅')

---
# PARTIE 1 — Le neurone et les fonctions d'activation

Un neurone calcule une **somme pondérée** puis applique une **activation** :
$z = \sum_i w_i x_i + b$, puis $a = f(z)$.

In [ ]:
x = np.array([0.9, 0.8, 0.0])     # entrées
w = np.array([0.5, 0.4, -0.6])    # poids (appris)
b = 0.1                            # biais (appris)

z = np.dot(w, x) + b
print(f'Somme pondérée z = {z:.3f}')
for nom, f in [('sigmoïde', sigmoid), ('tanh', np.tanh), ('ReLU', relu)]:
    print(f'  activation {nom:9s} -> a = {f(z):.4f}')

### Les activations et leurs dérivées

La backpropagation multiplie les **dérivées** des activations. Observez leur maximum :
0,25 pour la sigmoïde, 1 pour tanh et ReLU. C'est l'origine du *gradient qui s'évanouit* (section 7 du cours).

In [ ]:
z = np.linspace(-5, 5, 400)
fonctions = [('Sigmoïde', sigmoid, d_sigmoid), ('Tanh', np.tanh, d_tanh), ('ReLU', relu, d_relu)]

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for k, (nom, f, df) in enumerate(fonctions):
    axes[0, k].plot(z, f(z), lw=2.5); axes[0, k].set_title(nom)
    axes[1, k].plot(z, df(z), lw=2.5, color='tab:orange')
    axes[1, k].set_title(f'Dérivée — max = {df(z).max():.2f}')
    for ax in axes[:, k]:
        ax.axhline(0, color='grey', lw=0.8); ax.axvline(0, color='grey', lw=0.8); ax.grid(alpha=.3)
axes[0, 0].set_ylabel('f(z)'); axes[1, 0].set_ylabel("f'(z)")
plt.tight_layout(); plt.show()

---
# PARTIE 2 — Forward propagation : l'exemple de l'email

Réseau **3 → 2 → 1** (sigmoïde partout). On veut détecter un spam ($y = 1$) à partir de :

| Feature | Signification | Valeur |
|---------|---------------|--------|
| $x_1$ | proportion de mots suspects | 0,9 |
| $x_2$ | nombre de liens (normalisé) | 0,8 |
| $x_3$ | expéditeur dans les contacts | 0,0 |

In [ ]:
x = np.array([0.9, 0.8, 0.0])
y = 1.0                                   # c'est un spam

# Poids initiaux (identiques au cours)
W1_init = np.array([[ 0.5, 0.4, -0.6],    # vers h1
                    [-0.3, 0.8,  0.2]])   # vers h2
b1_init = np.array([0.1, -0.1])
W2_init = np.array([0.7, -0.4])
b2_init = 0.05

n_params = W1_init.size + b1_init.size + W2_init.size + 1
print(f'Nombre de paramètres du réseau : {n_params}')

In [ ]:
def forward_email(W1, b1, W2, b2, x=x):
    z1 = W1 @ x + b1;   a1 = sigmoid(z1)     # couche cachée
    z2 = W2 @ a1 + b2;  y_hat = sigmoid(z2)  # sortie
    return z1, a1, z2, y_hat

z1, a1, z2, y_hat = forward_email(W1_init, b1_init, W2_init, b2_init)

print('Couche cachée')
print(f'  h1 : z = 0.5*0.9 + 0.4*0.8 + (-0.6)*0 + 0.1 = {z1[0]:.4f}  ->  a = {a1[0]:.4f}')
print(f'  h2 : z = -0.3*0.9 + 0.8*0.8 + 0.2*0 - 0.1   = {z1[1]:.4f}  ->  a = {a1[1]:.4f}')
print('Sortie')
print(f'  z = 0.7*{a1[0]:.4f} + (-0.4)*{a1[1]:.4f} + 0.05 = {z2:.4f}  ->  ŷ = {y_hat:.4f}')
print(f'\nProbabilité de spam estimée : {y_hat:.1%}  (vérité : spam)')

---
# PARTIE 3 — La loss (entropie croisée binaire)

$$L = -\big[y \log \hat{y} + (1-y)\log(1-\hat{y})\big]$$

In [ ]:
def bce(y_hat, y, eps=1e-12):
    return -(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))

print(f'Loss de notre réseau : {bce(y_hat, y):.4f}')

# Une prédiction confiante ET fausse coûte très cher
tableau = pd.DataFrame({'ŷ prédit (vrai spam)': [0.99, 0.9, 0.579, 0.5, 0.1, 0.01]})
tableau['loss'] = bce(tableau['ŷ prédit (vrai spam)'], 1.0).round(3)
tableau

---
# PARTIE 4 — Backpropagation pas à pas

1. Erreur de sortie (sigmoïde + entropie croisée) : $\delta^{[2]} = \hat{y} - y$
2. Gradients de sortie : $\partial L / \partial w^{[2]}_j = \delta^{[2]} \, a_j$ et $\partial L / \partial b^{[2]} = \delta^{[2]}$
3. Erreur cachée : $\delta_j = \delta^{[2]} \, w^{[2]}_j \, a_j(1-a_j)$
4. Gradients cachés : $\partial L / \partial W^{[1]}_{ji} = \delta_j \, x_i$

In [ ]:
def backward_email(x, y, a1, y_hat, W2):
    d2 = y_hat - y                       # 1. erreur de sortie
    dW2 = d2 * a1; db2 = d2              # 2. gradients de sortie
    d1 = d2 * W2 * a1 * (1 - a1)         # 3. erreur des neurones cachés
    dW1 = np.outer(d1, x); db1 = d1      # 4. gradients cachés
    return d2, dW2, db2, d1, dW1, db1

d2, dW2, db2, d1, dW1, db1 = backward_email(x, y, a1, y_hat, W2_init)

print(f'δ sortie = ŷ - y = {y_hat:.4f} - 1 = {d2:.4f}')
print(f'∂L/∂w2 = {dW2}   ∂L/∂b2 = {db2:.4f}')
print(f'δ cachés = {d1}')

pd.DataFrame(dW1, index=['vers h1', 'vers h2'],
             columns=['depuis x1=0.9', 'depuis x2=0.8', 'depuis x3=0.0']).round(4)

> 🔑 La colonne **x3 = 0** a des gradients **nuls** : une connexion qui n'a rien transmis pendant la forward
> n'est pas responsable de l'erreur, donc elle n'est pas corrigée par cet exemple.

---
# PARTIE 5 — Mise à jour et itération

$$w \leftarrow w - \eta \, \frac{\partial L}{\partial w} \qquad (\eta = 0{,}5)$$

In [ ]:
eta = 0.5
W1 = W1_init - eta * dW1;  b1 = b1_init - eta * db1
W2 = W2_init - eta * dW2;  b2 = b2_init - eta * db2

print('W2 :', W2_init, '->', W2)
print(f'b2 : {b2_init} -> {b2:.4f}')
print('W1 :'); print(W1)

y_hat_new = forward_email(W1, b1, W2, b2)[3]
print(f'\nAprès 1 itération : ŷ = {y_hat:.4f} -> {y_hat_new:.4f}   loss = {bce(y_hat, y):.4f} -> {bce(y_hat_new, y):.4f}')

In [ ]:
# On répète le cycle forward -> backward -> mise à jour 50 fois
W1, b1, W2, b2 = W1_init.copy(), b1_init.copy(), W2_init.copy(), b2_init
historique = []
for it in range(50):
    z1, a1, z2, y_hat = forward_email(W1, b1, W2, b2)
    historique.append(bce(y_hat, y))
    _, dW2, db2, _, dW1, db1 = backward_email(x, y, a1, y_hat, W2)
    W1 -= eta * dW1; b1 -= eta * db1
    W2 = W2 - eta * dW2; b2 -= eta * db2

plt.figure(figsize=(8, 4))
plt.plot(historique, lw=2.5, color='tab:orange')
for i in [0, 1, 5, 20, 49]:
    plt.scatter(i, historique[i], color='tab:orange', zorder=3)
    plt.annotate(f'{historique[i]:.3f}', (i, historique[i]), textcoords='offset points', xytext=(6, 6))
plt.xlabel('Itération'); plt.ylabel('Loss'); plt.title('La loss diminue à chaque cycle')
plt.grid(alpha=.3); plt.show()
print(f'Probabilité de spam finale : {forward_email(W1, b1, W2, b2)[3]:.1%}')

> ⚠️ Le réseau a appris **un seul** email. Pour généraliser, il faut des milliers d'exemples et un jeu de test séparé :
> c'est l'objet de la partie 6.

---
# PARTIE 6 — Un réseau complet from scratch

Réseau **2 → 16 (ReLU) → 1 (sigmoïde)** sur `make_moons` : deux lunes imbriquées qu'aucune droite ne sépare.
On traite tous les exemples à la fois (matrices), d'où la convention $Z = XW + b$.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y_all = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_all, test_size=0.2, random_state=42, stratify=y_all)
y_tr, y_te = y_tr.reshape(-1, 1), y_te.reshape(-1, 1)

plt.figure(figsize=(6, 4.5))
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr.ravel(), cmap='coolwarm', s=12)
plt.title('make_moons : non séparable linéairement'); plt.show()
print('X_tr :', X_tr.shape, '| y_tr :', y_tr.shape)

In [ ]:
def init_params(n_in=2, n_h=16, n_out=1, seed=42, zeros=False):
    rng = np.random.default_rng(seed)
    if zeros:                                    # ⚠️ mauvaise idée (partie 7)
        return {'W1': np.zeros((n_in, n_h)), 'b1': np.zeros((1, n_h)),
                'W2': np.zeros((n_h, n_out)), 'b2': np.zeros((1, n_out))}
    return {'W1': rng.normal(0, np.sqrt(2 / n_in), (n_in, n_h)), 'b1': np.zeros((1, n_h)),   # init He
            'W2': rng.normal(0, np.sqrt(2 / n_h), (n_h, n_out)), 'b2': np.zeros((1, n_out))}

def forward(X, p):
    Z1 = X @ p['W1'] + p['b1'];   A1 = relu(Z1)
    Z2 = A1 @ p['W2'] + p['b2'];  A2 = sigmoid(Z2)
    return {'Z1': Z1, 'A1': A1, 'A2': A2}          # cache pour la backward

def loss_fn(A2, y, eps=1e-12):
    return -np.mean(y * np.log(A2 + eps) + (1 - y) * np.log(1 - A2 + eps))

def backward(X, y, p, c):
    m = X.shape[0]
    dZ2 = (c['A2'] - y) / m                         # δ sortie = ŷ - y
    dA1 = dZ2 @ p['W2'].T                           # l'erreur remonte
    dZ1 = dA1 * d_relu(c['Z1'])
    return {'W2': c['A1'].T @ dZ2, 'b2': dZ2.sum(axis=0, keepdims=True),
            'W1': X.T @ dZ1,       'b1': dZ1.sum(axis=0, keepdims=True)}

p = init_params()
for k, v in p.items():
    print(f'{k} : {v.shape}')
print('Total paramètres :', sum(v.size for v in p.values()))

### Gradient checking : notre backpropagation est-elle juste ?

On compare le gradient analytique à $\dfrac{L(w+\varepsilon) - L(w-\varepsilon)}{2\varepsilon}$ sur plusieurs paramètres de chaque matrice.

In [ ]:
def gradient_check(p, X, y, n_par_matrice=3, eps=1e-6, seed=0):
    rng = np.random.default_rng(seed)
    grads = backward(X, y, p, forward(X, p))
    lignes = []
    for nom in ['W1', 'b1', 'W2', 'b2']:
        for _ in range(n_par_matrice):
            idx = tuple(rng.integers(0, s) for s in p[nom].shape)
            original = p[nom][idx]
            p[nom][idx] = original + eps; L_plus  = loss_fn(forward(X, p)['A2'], y)
            p[nom][idx] = original - eps; L_moins = loss_fn(forward(X, p)['A2'], y)
            p[nom][idx] = original                        # on remet la valeur
            num, ana = (L_plus - L_moins) / (2 * eps), grads[nom][idx]
            ecart = abs(num - ana) / max(abs(num) + abs(ana), 1e-12)
            lignes.append({'paramètre': f'{nom}{list(idx)}', 'analytique': ana,
                           'numérique': num, 'écart relatif': ecart})
    return pd.DataFrame(lignes)

gc = gradient_check(init_params(), X_tr, y_tr)
print('Écart relatif maximal :', f"{gc['écart relatif'].max():.1e}", '-> OK ✅' if gc['écart relatif'].max() < 1e-7 else '-> à vérifier ⚠️')
gc

In [ ]:
def train(X, y, eta=0.5, epochs=2000, zeros=False, seed=42):
    p = init_params(zeros=zeros, seed=seed)
    pertes = []
    for _ in range(epochs):
        c = forward(X, p)                    # 1. forward
        pertes.append(loss_fn(c['A2'], y))   # 2. loss
        g = backward(X, y, p, c)             # 3. backward
        for k in p:                          # 4. mise à jour
            p[k] -= eta * g[k]
    return p, pertes

def accuracy(p, X, y):
    return ((forward(X, p)['A2'] > 0.5) == y).mean()

p, pertes = train(X_tr, y_tr, eta=0.5, epochs=2000)
for e in [0, 99, 499, 999, 1999]:
    print(f'epoch {e + 1:4d} | loss = {pertes[e]:.4f}')
print(f'\nAccuracy train : {accuracy(p, X_tr, y_tr):.3f} | test : {accuracy(p, X_te, y_te):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(pertes, lw=2); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Courbe de loss (noter le plateau avant la rechute)'); axes[0].grid(alpha=.3)

# Frontière de décision apprise
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
proba = forward(np.c_[xx.ravel(), yy.ravel()], p)['A2'].reshape(xx.shape)
axes[1].contourf(xx, yy, proba, levels=20, cmap='coolwarm', alpha=0.6)
axes[1].contour(xx, yy, proba, levels=[0.5], colors='black', linewidths=2)
axes[1].scatter(X_te[:, 0], X_te[:, 1], c=y_te.ravel(), cmap='coolwarm', edgecolor='k', s=18)
axes[1].set_title(f'Frontière de décision (test : {accuracy(p, X_te, y_te):.1%})')
plt.tight_layout(); plt.show()

> 🔑 La frontière (en noir) est **courbe** : grâce à la couche cachée et à ReLU, le réseau a appris une séparation
> qu'aucun modèle linéaire ne peut tracer.

---
# PARTIE 7 — Expériences

## 7.1 L'effet du learning rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for eta_test in [0.01, 0.5, 5.0, 50.0]:
    p_test, pertes_test = train(X_tr, y_tr, eta=eta_test, epochs=1000)
    hausses = int((np.diff(pertes_test) > 0).sum())
    ax.plot(pertes_test, lw=1.8, label=f'η = {eta_test:<5}  loss finale {pertes_test[-1]:.3f} | '
            f'hausses {hausses:>3}/999 | test {accuracy(p_test, X_te, y_te):.1%}')
ax.set_yscale('log')                     # échelle log : les pics d'η = 50 restent lisibles
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (échelle log)')
ax.set_title('Effet du learning rate'); ax.grid(alpha=.3, which='both')
ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2)   # légende sous le graphique
plt.tight_layout(); plt.show()

- **η = 0,01** : descente parfaitement régulière, mais lente : après 1 000 epochs, le modèle n'a pas fini d'apprendre.
- **η = 0,5** : descente régulière et efficace.
- **η = 5** : la loss **remonte près d'une epoch sur deux** (oscillations), mais elle finit basse : sur ce problème,
  un grand pas accélère malgré l'instabilité. Le « bon » learning rate dépend du problème : on le règle par essais.
- **η = 50** : **trop grand**. Les mises à jour dépassent sans cesse le minimum, la loss fait des sauts énormes
  et le modèle final est nettement moins bon.

## 7.2 Le problème de symétrie : initialiser à zéro

Si tous les poids valent 0, tous les neurones cachés calculent la même chose et reçoivent le même gradient.

In [ ]:
p_zero, pertes_zero = train(X_tr, y_tr, eta=0.5, epochs=1000, zeros=True)

print('Poids W1 après entraînement (initialisation à zéro) :')
print(p_zero['W1'][:, :5], '...')

colonnes_distinctes = np.unique(p_zero['W1'].round(8), axis=1).shape[1]   # 1 colonne = 1 neurone caché
print(f"\nTous les poids W1 sont-ils restés nuls ? {np.all(p_zero['W1'] == 0)}")
print(f'Neurones cachés distincts : {colonnes_distinctes} sur 16')
print(f'Loss finale : {pertes_zero[-1]:.4f} | accuracy test : {accuracy(p_zero, X_te, y_te):.1%}')

> 🔑 Avec ReLU et des poids nuls, $Z_1 = 0$ partout : la dérivée de ReLU vaut 0, aucun gradient n'atteint la couche cachée,
> et le réseau ne peut apprendre qu'une constante (≈ 50 % d'accuracy). Une initialisation **aléatoire** (He, Xavier)
> brise cette symétrie.

---
# PARTIE 8 — Le même réseau avec scikit-learn

`MLPClassifier` exécute en interne exactement nos étapes (forward, backprop), avec l'optimiseur **Adam**.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(16,), activation='relu', solver='adam',
                    max_iter=2000, random_state=42)
mlp.fit(X_tr, y_tr.ravel())

comparaison = pd.DataFrame({
    'modèle': ['NumPy from scratch (SGD batch)', 'scikit-learn MLPClassifier (Adam)'],
    'accuracy test': [accuracy(p, X_te, y_te), mlp.score(X_te, y_te.ravel())],
    'itérations': [len(pertes), mlp.n_iter_],
})
comparaison.round(3)

---
# ✅ Synthèse

| Étape | Ce que vous avez vu |
|-------|---------------------|
| **Forward** | $z = Wa + b$, $a = f(z)$, couche après couche jusqu'à $\hat{y}$ |
| **Loss** | l'entropie croisée punit fortement les prédictions confiantes et fausses |
| **Backward** | $\delta = \hat{y} - y$ en sortie, puis l'erreur remonte ; une entrée nulle donne un gradient nul |
| **Mise à jour** | $w \leftarrow w - \eta \, \partial L / \partial w$, répétée jusqu'à convergence |
| **Vérification** | le gradient checking valide une backprop écrite à la main |
| **Pièges** | learning rate mal réglé, initialisation à zéro (symétrie) |

### 🚀 Pour aller plus loin
- Remplacez ReLU par la sigmoïde dans la couche cachée (`d_sigmoid`) et relancez le gradient checking.
- Passez la couche cachée à 256 neurones avec seulement 100 exemples d'entraînement : observez le surapprentissage.
- Ajoutez une seconde couche cachée et écrivez sa backpropagation.

---
*📘 Module Data Science — Forward et Backward Propagation | Bootcamp Data Science*